# Augplot: start with auto, then add the hard part in one sentence

Two examples where a short refinement replaces substantial plotting code: highlight CV winners with fold variability, then turn model outputs into a forecast diagnostic. All data here is synthetic.

**Setup:** install using the [README](../README.md#install), select your Augplot kernel, and run the first cell. It selects `openai/gpt-5.6-terra` and asks for your API key with hidden input.

First run: four generations, plus one if you enable Plotly; each may need one repair request. Identical reruns use saved code. Your provider receives a data profile; generated Python runs locally and is not sandboxed.

In [ ]:
import os
from getpass import getpass

os.environ["AUGPLOT_MODEL"] = "openai/gpt-5.6-terra"
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

In [ ]:
import numpy as np
import pandas as pd

import augplot as ap

## 1. Start from an automatic CV comparison

Five classifiers, five CV folds, and two higher-is-better metrics. Pass the results directly to `ap.plot()` and let Augplot choose the first chart.

The next cell adds the analysis-aware details that usually take most of the plotting code.

In [ ]:
results_dict = {
    "Logistic regression": {
        "accuracy": np.array([0.80, 0.82, 0.81, 0.79, 0.84]),
        "roc_auc": np.array([0.88, 0.90, 0.89, 0.86, 0.90]),
    },
    "Random forest": {
        "accuracy": np.array([0.88, 0.90, 0.89, 0.86, 0.90]),
        "roc_auc": np.array([0.95, 0.96, 0.94, 0.93, 0.95]),
    },
    "Gradient boosting": {
        "accuracy": np.array([0.92, 0.93, 0.91, 0.92, 0.93]),
        "roc_auc": np.array([0.94, 0.95, 0.93, 0.92, 0.95]),
    },
    "Extra trees": {
        "accuracy": np.array([0.86, 0.88, 0.89, 0.87, 0.88]),
        "roc_auc": np.array([0.91, 0.93, 0.92, 0.90, 0.92]),
    },
    "Neural network": {
        "accuracy": np.array([0.94, 0.88, 0.93, 0.84, 0.95]),
        "roc_auc": np.array([0.96, 0.90, 0.95, 0.86, 0.97]),
    },
}

viz = ap.plot(results_dict, backend="matplotlib")

In [ ]:
print(viz.explanation)
print("Reused saved code:", viz.cache_hit)

## 2. Add the hard part in one sentence

This refinement has to aggregate folds, find a different winner for each metric, preserve variability, style individual bars, place labels above error bars, and update the legend. “Best” means highest observed mean, not statistical significance.

In [ ]:
viz.refine("Highlight the best model for each metric and show fold variability.")

In [ ]:
# The generated function and its saved version are available for inspection.
print("Saved source:", viz.history_path)
# print(viz.code)

## 3. New results, same function

These synthetic updated scores make the neural network the accuracy winner. `render()` should move the highlight automatically, without calling the LLM. Export the current function under a readable name when you're happy with it.

In [ ]:
updated_results = {
    model: {metric: values.copy() for metric, values in metrics.items()}
    for model, metrics in results_dict.items()
}
updated_results["Neural network"]["accuracy"] = np.array([0.947, 0.943, 0.952, 0.938, 0.950])
viz.render(updated_results, title="Updated CV results — a new accuracy winner")

In [ ]:
from pathlib import Path

# Choose a fresh filename on reruns so we do not overwrite existing code.
export_path = Path("vis_utils.py")
version = 2
while export_path.exists():
    export_path = Path(f"vis_utils_{version}.py")
    version += 1

viz.save(export_path, function_name="plot_cv_results")

In [ ]:
# The printed snippet works for normal imports. Here we load the chosen file
# directly so this cell also works when a rerun selected a numbered filename.
import importlib.util

spec = importlib.util.spec_from_file_location(export_path.stem, export_path)
vis_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vis_utils)

fig = vis_utils.plot_cv_results(updated_results, figsize=(11, 5))
fig

## 4. Start from observed model-serving latency

Suppose an upstream model forecasts weekly inference latency. The table contains observed latency, supplied forecasts, and interval bounds. Six forecast weeks now have observations; the last two are still in the future.

Start with the observations. Augplot does not fit the forecasting model or calculate the interval.

In [ ]:
rng = np.random.default_rng(42)
weeks = pd.date_range("2025-01-06", periods=32, freq="W-MON")
history = pd.DataFrame({
    "week": weeks[:24],
    "actual": np.round(450 + 9 * np.arange(24) + rng.normal(0, 30, 24)),
    "forecast": np.nan,
    "lower": np.nan,
    "upper": np.nan,
})
# Fixed stand-ins for predictions/intervals from an upstream model. No training here.
forecast_window = pd.DataFrame({
    "week": weeks[24:],
    "actual": [675, 710, 620, 713, 810, 728, np.nan, np.nan],
    "forecast": [668, 691, 679, 708, 734, 720, 755, 773],
    "lower": [628, 648, 634, 660, 684, 668, 699, 715],
    "upper": [708, 734, 724, 756, 784, 772, 811, 831],
})
forecast_results = pd.concat([history, forecast_window], ignore_index=True)

forecast_viz = ap.plot(
    forecast_results,
    backend="seaborn",
    prompt="Plot observed inference latency over time.",
)

## 5. Turn it into a forecast diagnostic in one sentence

A useful diagnostic needs several coordinated layers: forecast, interval band, forecast boundary, missing future observations, and interval misses. The prompt stays short because those details are already present in the data.

In [ ]:
forecast_viz.refine("Overlay the forecast and interval, mark where forecasting starts, and highlight interval misses.")

## 6. Optional: make the diagnostic interactive

Set the toggle to `True` to generate the same diagnostic with Plotly hover details.

In [ ]:
RUN_PLOTLY = False

if RUN_PLOTLY:
    interactive_viz = ap.plot(
        forecast_results,
        prompt="Show the forecast diagnostic with hover details for every model output.",
        backend="plotly",
    )

**Rerunning:** run from the original `ap.plot()` cell to replay the same refinement. Repeating only `refine()` edits the current version again. Keep `.augplot/` with this notebook. [How history works](../docs/visualization-history.md).